# 04 — Build each client's local training table (extended history)

Turns `rtt_long.csv` into the table every federated client trains on:
one row per `(trust, specialty, target month)` → features + next-month demand target.
Output: `data/processed/features.csv` (clients slice it by `provider_code`).

**Design choices (each with its reason):**
- **Features**: demand lags 1/2/3/12 (12 = same month last year → seasonality), backlog and
  throughput at t−1 (the flow-identity context), month-of-year as sin/cos (cyclic).
  All features are values known *before* the target month — nothing from the future.
- **Frozen panel (pre-registered)**: series = those complete on 2021-04..2026-03 (the same 694
  as the laptop run). Pre-2018-21 months only add history — they never change the panel.
- **Per-series normalization**: z-score of log1p, stats computed on **train months only**.
  Kills the 3.4× trust-size spread (non-IID) so all series train on a comparable scale.
- **Time split, never random**: targets 2022-04..2024-09 train (30 mo) · 2024-10..2025-03 val (6 mo)
  · 2025-04..2026-03 test (12 mo). First target is 2022-04 because lag-12 needs a year of history.

In [1]:
import numpy as np
import pandas as pd

long = pd.read_csv("../data/processed/rtt_long.csv")

# one column per operational measure: demand (Part_3), backlog (Part_2), throughput (1A+1B)
key = ["provider_code", "specialty_code", "month"]
cnt = long.pivot_table(index=key, columns="part", values="total_all", aggfunc="sum").reset_index()
wide = pd.DataFrame({
    "provider_code": cnt["provider_code"], "specialty_code": cnt["specialty_code"], "month": cnt["month"],
    "demand": cnt["Part_3"], "backlog": cnt["Part_2"],
    "throughput": cnt["Part_1A"].fillna(0) + cnt["Part_1B"].fillna(0),
})

# complete grid: every kept series x all 60 months (missing backlog/throughput months -> 0)
months = sorted(long["month"].unique())
print(f"months: {len(months)} | raw (trust, specialty, month) rows: {len(wide)}")

months: 60 | raw (trust, specialty, month) rows: 44371


In [2]:
# FROZEN PANEL (pre-registered): series kept = complete on the 2021-04..2026-03 window
# (the same 694 as the laptop run). Pre-2021 months are extra history only — they can add
# training rows but can never add or remove a series.
WINDOW = ("2021-04", "2026-03")
core = wide[wide["month"].between(*WINDOW)]
n_win = core["month"].nunique()
ok = core.dropna(subset=["demand"]).query("demand > 0").groupby(["provider_code", "specialty_code"])["month"].nunique()
keep = ok[ok == n_win].index
wide = wide.set_index(["provider_code", "specialty_code"]).loc[keep].reset_index()
wide[["backlog", "throughput"]] = wide[["backlog", "throughput"]].fillna(0)
print(f"series kept: {len(keep)} of {len(ok)} (dropped {len(ok) - len(keep)} incomplete)")
print(f"per trust: min {keep.to_frame(index=False).groupby('provider_code').size().min()}, "
      f"max {keep.to_frame(index=False).groupby('provider_code').size().max()} series")

series kept: 694 of 760 (dropped 66 incomplete)
per trust: min 16, max 23 series


In [3]:
# per-series features on the log scale; lags shift within each series (sorted by month)
wide = wide.sort_values(["provider_code", "specialty_code", "month"]).reset_index(drop=True)
g = wide.groupby(["provider_code", "specialty_code"], sort=False)
f = wide[key].copy()
f["demand"] = wide["demand"]                       # raw target month value (kept for evaluation)
f["y"] = np.log1p(wide["demand"])                  # model target (log scale)
for lag in [1, 2, 3, 12]:
    f[f"lag{lag}"] = g["demand"].shift(lag).pipe(np.log1p)
f["backlog"] = g["backlog"].shift(1).pipe(np.log1p)      # known at t-1
f["throughput"] = g["throughput"].shift(1).pipe(np.log1p)
mnum = f["month"].str[5:7].astype(int)
f["month_sin"] = np.sin(2 * np.pi * mnum / 12)
f["month_cos"] = np.cos(2 * np.pi * mnum / 12)

f = f.dropna().reset_index(drop=True)              # lag-12 warmup: drops each series' first 12 observed months
f = f[f["demand"] > 0].reset_index(drop=True)      # pre-2021 history may contain zero months for some series — not valid targets

# time split by target month
f["split"] = np.where(f["month"] <= "2024-09", "train", np.where(f["month"] <= "2025-03", "val", "test"))
print(f.groupby("split")["month"].agg(["min", "max", "count"]))

           min      max  count
split                         
test   2025-04  2026-03   8328
train  2022-04  2024-09  20820
val    2024-10  2025-03   4164


In [4]:
# per-series z-score, stats from TRAIN months only (no leakage into val/test)
ser = ["provider_code", "specialty_code"]
tr = f[f["split"] == "train"]
stats = tr.groupby(ser).agg(mu=("y", "mean"), sd=("y", "std"),
                            mu_b=("backlog", "mean"), sd_b=("backlog", "std"),
                            mu_t=("throughput", "mean"), sd_t=("throughput", "std")).reset_index()
stats[["sd", "sd_b", "sd_t"]] = stats[["sd", "sd_b", "sd_t"]].clip(lower=1e-6)  # guard flat series
f = f.merge(stats, on=ser)

f["y"] = (f["y"] - f["mu"]) / f["sd"]                    # standardized target
for c in ["lag1", "lag2", "lag3", "lag12"]:
    f[c] = (f[c] - f["mu"]) / f["sd"]                    # lags share the demand stats
f["backlog"] = (f["backlog"] - f["mu_b"]) / f["sd_b"]
f["throughput"] = (f["throughput"] - f["mu_t"]) / f["sd_t"]
f = f.drop(columns=["mu_b", "sd_b", "mu_t", "sd_t"])     # keep mu/sd: needed to invert predictions

f.to_csv("../data/processed/features.csv", index=False)
print(f"saved features.csv: {len(f)} rows x {f.shape[1]} cols")
f.head(3)

saved features.csv: 33312 rows x 16 cols


,provider_code,specialty_code,month,demand,y,lag1,lag2,lag3,lag12,backlog,throughput,month_sin,month_cos,split,mu,sd
0,R0B,C_100,2022-04,1494.0,-1.504557,0.472124,-1.347408,-2.063336,-1.176922,-1.792877,-0.996174,8.660254e-01,-0.500000,train,7.437114,0.084564
1,R0B,C_100,2022-05,1831.0,0.899313,-1.504557,0.472124,-1.347408,-1.192320,-1.825048,-1.527373,5.000000e-01,-0.866025,train,7.437114,0.084564
2,R0B,C_100,2022-06,1693.0,-0.026793,0.899313,-1.504557,0.472124,-0.174303,-0.771157,-0.395262,1.224647e-16,-1.000000,train,7.437114,0.084564


In [5]:
# sanity checks
assert not f.drop(columns="month").isna().any().any(), "NaNs in features"

# 1. lag alignment: lag1 of month t must equal the (standardized log) demand of month t-1
s = f[(f["provider_code"] == "RAJ") & (f["specialty_code"] == "C_130")].sort_values("month")
assert np.allclose(s["lag1"].values[1:], s["y"].values[:-1]), "lag misaligned"

# 2. leakage: normalization stats must reproduce from train rows alone
chk = f[f["split"] == "train"].groupby(ser)["y"].agg(["mean", "std"])
assert np.allclose(chk["mean"], 0, atol=1e-6) and np.allclose(chk["std"], 1, atol=1e-3), "stats not train-only"

# 3. every series contributes to every split
n_splits = f.groupby(ser)["split"].nunique()
assert (n_splits == 3).all(), "series missing a split"

print(f"all checks pass | series: {f.groupby(ser).ngroups} | trusts: {f['provider_code'].nunique()} "
      f"| rows train/val/test: {(f['split'] == 'train').sum()}/{(f['split'] == 'val').sum()}/{(f['split'] == 'test').sum()}")

all checks pass | series: 694 | trusts: 35 | rows train/val/test: 20820/4164/8328
